# 02. 가게 생존 — 개업 후 2년 안에 폐업할 확률 (LightGBM)

**질문**: 이 가게는 문 연 뒤 2년 안에 닫을까?

- 입력 파일: `data/survival_v1.parquet` (행 = 가게 1곳), `data/survival_v1_spec.json` (피처 목록)
- 가게 정보는 행정안전부 인허가 데이터(일반·휴게음식점)에서 왔고, 각 가게에 좌표로 상권을 붙이고(경계 안 또는 300m 이내) 업태를 서울시 업종 코드로 바꿔서, **개업한 분기의 상권×업종 수치**(01의 피처)를 함께 붙였다.
- 라벨 `y_2y` = 개업일로부터 730일 안에 폐업하면 1. 피처는 **개업일 이전에 알 수 있는 정보만** (면적·업태·상호 특징·같은 자리의 이전 가게 이력·반경 100/300m 경쟁·지하철 거리·개업 분기 상권 컨텍스트).
- 분할은 개업연도: train 2021 / val 2022 / test 2023. 2024년 이후 개업은 2년이 안 지나 라벨이 없어 뺀다.

In [1]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

DATA = next(p for p in [Path("data"), Path("../data/processed")] if (p / "survival_v1.parquet").exists())
df = pd.read_parquet(DATA / "survival_v1.parquet")
spec = json.load(open(DATA / "survival_v1_spec.json", encoding="utf-8"))
NUM, CAT = spec["features_num"], spec["features_cat"]          # 수치 103개, 범주 9개

# 주 코호트: 2021~2023 개업 + 정제 (상호에 '한시적·팝업' 같은 임시 허가, 면적 0, 개업 30일 내 폐업은 창업이 아니라서 제외)
df = df[df["cohort_main"] & df["split"].isin(["train", "val", "test"])].reset_index(drop=True)
print(df.shape, "| 분할:", df["split"].value_counts().to_dict())
print("2년 폐업률:", df.groupby("split")["y_2y"].mean().round(3).to_dict())
df[["개업연도", "업태", "업종_코드", "면적", "y_2y", "split"]].head()

(53964, 145) | 분할: {'train': 18631, 'test': 18278, 'val': 17055}
2년 폐업률: {'test': 0.262, 'train': 0.262, 'val': 0.267}


,개업연도,업태,업종_코드,면적,y_2y,split
0,2022,기타,CS100006,42.80,0.0,val
1,2022,식육(숯불구이),CS100001,71.92,1.0,val
2,2023,경양식,CS100004,68.77,0.0,test
3,2021,한식,CS100001,22.68,1.0,train
4,2021,한식,CS100001,30.00,1.0,train


## 1. 피처 준비와 분할

In [2]:
tr, va, te = (df["split"] == s for s in ("train", "val", "test"))
y = df["y_2y"].astype(int)

X = df[NUM + CAT].copy()
for c in CAT:
    cats = sorted(df[c].dropna().astype(str).unique())
    X[c] = pd.Categorical(X[c].astype(str).where(X[c].notna(), None), categories=cats)
print("train/val/test:", tr.sum(), va.sum(), te.sum(), "| 피처", X.shape[1], "개")

train/val/test: 18631 17055 18278 | 피처 112 개


## 2. 기준선 — 업태 × 자치구 평균 폐업률
train(2021 개업)에서 "강남구 한식집은 몇 %가 닫았나"를 그대로 예측값으로 쓴다. 표본이 30곳 미만이면 업태 평균 → 전체 평균으로 물러난다.

In [3]:
def rate_baseline(keys, min_n=30):
    train = df[tr]
    pred = pd.Series(np.nan, index=df.index)
    for k in range(len(keys), 0, -1):                                   # 세부 그룹부터 → 표본 부족하면 상위 그룹으로
        g = train.groupby(keys[:k])["y_2y"].agg(["mean", "size"])
        ok = g[g["size"] >= min_n]["mean"]
        fill = df[keys[:k]].apply(tuple, axis=1).map(ok) if k > 1 else df[keys[0]].map(ok)
        pred = pred.where(pred.notna(), fill)
    return pred.fillna(train["y_2y"].mean())

p_base = rate_baseline(["업태", "자치구_코드"])
print("기준선 test AUC:", round(roc_auc_score(y[te], p_base[te]), 4))

기준선 test AUC: 0.5428


## 3. LightGBM 학습 (val 조기종료)

In [4]:
PARAMS = dict(objective="binary", metric="auc", learning_rate=0.05, num_leaves=31, min_data_in_leaf=100,
              feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=255, verbose=-1, seed=42, num_threads=8)

model = lgb.train(PARAMS, lgb.Dataset(X[tr], y[tr]), num_boost_round=5000, valid_sets=[lgb.Dataset(X[va], y[va])],
                  callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
p = pd.Series(model.predict(X, num_iteration=model.best_iteration), index=df.index)
print("멈춘 라운드:", model.best_iteration)

멈춘 라운드: 53


## 4. 결과 (test = 2023년 개업 가게)
- **AUC**: 폐업한 가게와 아닌 가게를 얼마나 잘 구분하나 (0.5 = 동전 던지기).
- **위험 상위 20% 적중률**: 모델이 가장 위험하다고 한 20% 가게 중 실제로 닫은 비율. 전체 폐업률과 비교해서 본다.

In [5]:
def metrics(y_true, prob, top=0.2):
    yv, pv = np.asarray(y_true, float), np.asarray(prob, float)
    k = int(round(len(yv) * top)); idx = np.argsort(-pv)[:k]
    return {"n": len(yv), "실제 폐업률": yv.mean(), "AUC": roc_auc_score(yv, pv), "PR-AUC": average_precision_score(yv, pv),
            "상위20% 적중률": yv[idx].mean(), "lift": yv[idx].mean() / yv.mean()}

pd.DataFrame({"기준선 (업태×자치구)": metrics(y[te], p_base[te]), "LightGBM": metrics(y[te], p[te])}).T.round(3)

,n,실제 폐업률,AUC,PR-AUC,상위20% 적중률,lift
기준선 (업태×자치구),18278.0,0.262,0.543,0.287,0.300,1.148
LightGBM,18278.0,0.262,0.669,0.432,0.446,1.705


## 5. 보정 — 예측 확률 10분위별 실제 폐업률 (직선에 가까울수록 확률을 믿을 수 있다)

In [6]:
calib = pd.DataFrame({"pred": p[te], "y": y[te]})
calib["분위"] = pd.qcut(calib["pred"], 10, labels=False) + 1
calib.groupby("분위").agg(n=("y", "size"), 예측_평균=("pred", "mean"), 실제_폐업률=("y", "mean")).round(3)

,n,예측_평균,실제_폐업률
분위,,,
1,1828,0.125,0.109
2,1828,0.156,0.147
3,1828,0.181,0.171
4,1827,0.205,0.198
5,1828,0.227,0.209
6,1828,0.252,0.269
7,1827,0.280,0.281
8,1828,0.313,0.340
9,1829,0.356,0.374


## 6. 어떤 피처가 중요했나 (gain 기준 상위 15)

In [7]:
imp = pd.Series(model.feature_importance("gain"), index=model.feature_name())
(imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(2).to_frame("중요도 %")

,중요도 %
log_area,23.73
자치구_코드,9.52
site_n_prior,3.72
업태,3.58
nm_branch,2.85
nb100_opened_1y,2.71
site_closures_5y,2.24
site_prev_dur_days,2.20
subway_dist_m,2.04
nb100_same_upte,1.71


## 읽는 법
- 기대값(프로젝트 기록, test): 기준선 AUC 0.543 → LightGBM **0.669**, 위험 상위 20%의 실제 폐업률 약 45% (전체 26%).
- 면적이 가장 큰 신호(작을수록 위험), 그다음 같은 자리의 이전 가게 이력·주변 신규 개업 수.
- **하지 말 것**: 폐업일·폐업 여부에서 파생된 값을 피처에 넣는 것(정답 누수), 개업연도를 섞은 무작위 분할.